
# Scala Functional Error Handling — Deep Dive

During this lecture, we will cover:

- Why Functional Error Handling?
- `Option[A]` — Modeling the Absence of a Value
- `Try[A]` — Capturing Exceptions as Values
- `Either[E, A]` — Typed, Informative Error Channels
- Transforming: `map`, `flatMap`, `fold`, `getOrElse`
- Chaining and Composing Error-Prone Operations
- `for`-Comprehensions over `Option`, `Try`, `Either`
- Converting Between `Option`, `Try`, and `Either`
- Accumulating Multiple Errors
- Building a Complete Data Processing Pipeline with Pure Functions

These concepts form the **backbone of safe, composable Scala code**. They appear everywhere in production Scala systems — from Apache Spark data pipelines, to Akka HTTP request handling, to Cats-based domain logic.

Mastering them allows you to **eliminate null pointer exceptions**, make error paths **explicit and type-checked**, and write code that is **safe, readable, and composable** — without a single `try/catch` block in your business logic.



# 1. Why Functional Error Handling?

In traditional object-oriented code, errors are typically communicated in two ways that are both problematic:

1. **`null` returns** — The callee silently returns `null` to signal absence. The caller forgets to check, and a `NullPointerException` explodes at runtime, far from the original source.
2. **Thrown exceptions** — The callee `throw`s an exception. The type signature of the function says nothing about this possibility. The caller must remember to `try/catch`, or the exception propagates up the call stack unpredictably.

Both approaches violate a core principle of functional programming: **functions should be total** — they should return a value for every input, and that value should honestly describe the outcome.

Functional error handling solves this by making errors **first-class values** in the return type:

| Problem | FP Solution | Type |
|---------|------------|------|
| Value may be absent | Represent it explicitly | `Option[A]` |
| Computation may throw | Capture the exception | `Try[A]` |
| Computation may fail with a typed reason | Return a choice | `Either[E, A]` |

The compiler then **forces the caller** to handle both the success and failure case — making error handling impossible to accidentally forget.


In [ ]:
// The problem with null
def findUserUnsafe(id: Int): String = {
  val db = Map(1 -> "Alice", 2 -> "Bob")
  db.getOrElse(id, null)  // returns null if not found
}

// This compiles and runs — until it explodes
val name = findUserUnsafe(99)
// name.toUpperCase  // NullPointerException at runtime!

// The problem with unchecked exceptions
def parseAgeUnsafe(s: String): Int = s.toInt  // throws NumberFormatException

// The type says Int => Int. Nothing tells you it can blow up.
// parseAgeUnsafe("twenty")  // exception!

println("Both patterns compile fine, but fail unpredictably at runtime.")
println("Functional error handling makes failure visible in the type signature.")


# 2. `Option[A]` — The Presence or Absence of a Value

`Option[A]` is a sealed ADT with exactly two cases:

- `Some(value: A)` — a value is present
- `None` — no value is present

It is the **type-safe replacement for `null`**. Instead of returning `null` to signal absence, a function returns `None`. The type system then forces every caller to handle both cases.

```scala
sealed trait Option[+A]
case class Some[A](value: A) extends Option[A]
case object None             extends Option[Nothing]
```

The `+A` covariance annotation means `Option[Nothing]` (i.e., `None`) is a subtype of `Option[A]` for any `A` — so `None` can be used wherever any `Option` is expected.


In [ ]:
// Creating Option values
val present: Option[Int]    = Some(42)
val absent:  Option[Int]    = None
val fromNull: Option[String] = Option(null)   // Option(null) => None
val fromVal:  Option[String] = Option("hello") // Option("hello") => Some("hello")

present
absent
fromNull
fromVal

In [ ]:
// Safe lookup — returns Option instead of null
def findUser(id: Int): Option[String] = {
  val db = Map(1 -> "Alice", 2 -> "Bob", 3 -> "Charlie")
  db.get(id)  // Map.get already returns Option
}

findUser(1)    // Some("Alice")
findUser(99)   // None

// Map.get is itself Option-based
val scores = Map("Alice" -> 95, "Bob" -> 72)
scores.get("Alice")   // Some(95)
scores.get("Diana")   // None


## 2.1 Extracting Values from `Option`

There are several ways to extract a value from an `Option`, ranging from unsafe to fully safe.


In [ ]:
val maybeScore: Option[Int] = Some(88)
val noScore:    Option[Int] = None

// getOrElse — provide a fallback default
maybeScore.getOrElse(0)   // 88
noScore.getOrElse(0)      // 0

// orElse — fallback to another Option
noScore.orElse(Some(50))  // Some(50)
noScore.orElse(None)      // None

// fold — provide both the None case and the Some case
maybeScore.fold("No score")(s => s"Score: $s")  // "Score: 88"
noScore.fold("No score")(s => s"Score: $s")     // "No score"

// Pattern matching — the most explicit approach
maybeScore match {
  case Some(s) => s"Found: $s"
  case None    => "Not found"
}

In [ ]:
// isDefined / isEmpty — for conditional logic (but prefer map/fold)
maybeScore.isDefined   // true
noScore.isEmpty        // true

// get — UNSAFE, throws NoSuchElementException on None
// Only use when you have already verified isDefined — or better, don't use it
maybeScore.get         // 88
// noScore.get         // throws NoSuchElementException!


## 2.2 Transforming `Option` with `map`, `flatMap`, `filter`

The real power of `Option` is that it supports `map`, `flatMap`, and `filter` — the same HOFs available on collections.

This means you can **transform and chain optional computations** without ever manually unwrapping the value or writing `if (opt.isDefined)` checks.

- `map` — transform the value inside `Some`, propagate `None`
- `flatMap` — chain a function that itself returns `Option`
- `filter` — turn `Some(x)` into `None` if `x` doesn't satisfy the predicate


In [ ]:
val score: Option[Int] = Some(75)
val noVal: Option[Int] = None

// map — transform the value; None passes through unchanged
score.map(_ * 2)            // Some(150)
noVal.map(_ * 2)            // None

score.map(s => s"Grade: $s%")
noVal.map(s => s"Grade: $s%")

// filter — Some becomes None if predicate fails
score.filter(_ >= 60)       // Some(75) — passes
score.filter(_ >= 80)       // None — fails, 75 < 80
noVal.filter(_ >= 60)       // None — stays None

In [ ]:
// flatMap — chain optional operations without nested Some(Some(...))
def findDepartment(userId: Int): Option[String] =
  Map(1 -> "Engineering", 2 -> "Marketing").get(userId)

def findManager(dept: String): Option[String] =
  Map("Engineering" -> "Carol", "Marketing" -> "Dave").get(dept)

// Without flatMap — manual, ugly, easy to forget a case
val manualResult: Option[String] = findDepartment(1) match {
  case None       => None
  case Some(dept) => findManager(dept)
}

// With flatMap — clean and declarative
val manager1 = findDepartment(1).flatMap(findManager)  // Some("Carol")
val manager2 = findDepartment(2).flatMap(findManager)  // Some("Dave")
val manager3 = findDepartment(9).flatMap(findManager)  // None — no user 9
val manager4 = findDepartment(1).flatMap(_ => findManager("HR"))  // None — no HR manager

manager1
manager2
manager3
manager4


## 2.3 `for`-Comprehensions over `Option`

When you need to chain multiple `flatMap`s on `Option`, a `for`-comprehension produces much more readable code.

Each `<-` in a `for`-comprehension over `Option` means: "unwrap this `Some`, or short-circuit to `None` if it's `None`."


In [ ]:
case class User(id: Int, name: String, addressId: Int)
case class Address(id: Int, city: String, postcode: String)
case class Weather(city: String, description: String, tempC: Double)

val users    = Map(1 -> User(1, "Alice", 10), 2 -> User(2, "Bob", 20))
val addresses = Map(10 -> Address(10, "Istanbul", "34000"), 20 -> Address(20, "Ankara", "06000"))
val weather  = Map("Istanbul" -> Weather("Istanbul", "Sunny", 22.0))

// Chain three lookups using for-comprehension
def userWeather(userId: Int): Option[String] =
  for {
    user    <- users.get(userId)
    address <- addresses.get(user.addressId)
    wx      <- weather.get(address.city)
  } yield s"${user.name} in ${address.city}: ${wx.description}, ${wx.tempC}°C"

userWeather(1)   // Some — all three lookups succeed
userWeather(2)   // None — Ankara has no weather entry
userWeather(9)   // None — user 9 doesn't exist

In [ ]:
// for-comprehension with guards
def highScorer(userId: Int): Option[String] = {
  val scores = Map(1 -> 95, 2 -> 62, 3 -> 88)
  val names  = Map(1 -> "Alice", 2 -> "Bob", 3 -> "Charlie")
  for {
    score <- scores.get(userId)
    if score >= 85          // guard: short-circuits to None if score < 85
    name  <- names.get(userId)
  } yield s"$name scored $score — distinction!"
}

highScorer(1)   // Some — score 95 >= 85
highScorer(2)   // None — score 62 < 85 (guard fails)
highScorer(3)   // Some — score 88 >= 85
highScorer(9)   // None — user not found


## 2.4 `Option` in Data Processing

`Option` is indispensable when processing real-world data that may have missing fields, failed lookups, or optional attributes.


In [ ]:
case class RawRecord(name: String, ageStr: String, emailOpt: Option[String])

// Safe parsing using Option
def parseAge(s: String): Option[Int] =
  s.trim.toIntOption   // built-in since Scala 2.13

def validateEmail(email: String): Option[String] =
  if (email.contains("@") && email.contains(".")) Some(email.trim.toLowerCase)
  else None

case class ValidRecord(name: String, age: Int, email: String)

def validateRecord(raw: RawRecord): Option[ValidRecord] =
  for {
    age   <- parseAge(raw.ageStr)
    if age >= 0 && age <= 150
    email <- raw.emailOpt
    valid <- validateEmail(email)
  } yield ValidRecord(raw.name.trim.capitalize, age, valid)

val records = List(
  RawRecord("alice",   "30",     Some("alice@example.com")),
  RawRecord("bob",     "twenty", Some("bob@example.com")),  // bad age
  RawRecord("charlie", "28",     None),                    // no email
  RawRecord("diana",   "25",     Some("not-an-email")),    // bad email
  RawRecord("eve",     "35",     Some("EVE@Corp.COM"))      // valid
)

val results = records.map(validateRecord)

// Separate valid from invalid
val valid   = results.flatten          // collect only Some values
val invalid = results.count(_.isEmpty) // count None values

println(s"Valid: $valid")
println(s"Invalid count: $invalid")


Note: `List[Option[A]].flatten` is equivalent to `list.flatMap(identity)` — it keeps only the `Some` values and discards `None`s. This is a very common pattern for processing optional results in bulk.



# 3. `Try[A]` — Capturing Exceptions as Values

`Try[A]` is a sealed ADT that represents a computation that **either succeeded with a value or failed with an exception**:

- `Success(value: A)` — computation completed normally
- `Failure(exception: Throwable)` — computation threw an exception

```scala
sealed trait Try[+A]
case class Success[A](value: A)          extends Try[A]
case class Failure[A](exception: Throwable) extends Try[A]
```

`Try` is designed specifically to **wrap code that throws exceptions** — such as parsing, I/O, reflection, or third-party Java libraries. It captures the exception as a value instead of letting it propagate.

The key constructor is `Try { expression }` — it evaluates the expression and wraps it in `Success` or `Failure` depending on whether it throws.


In [ ]:
import scala.util.{Try, Success, Failure}

// Try captures exceptions as values
val good: Try[Int]  = Try("42".toInt)       // Success(42)
val bad:  Try[Int]  = Try("abc".toInt)      // Failure(NumberFormatException)
val zero: Try[Int]  = Try(10 / 0)           // Failure(ArithmeticException)

good
bad
zero

In [ ]:
// Extracting values from Try
val result = Try("99".toInt)

// getOrElse — provide default on failure
result.getOrElse(0)                   // 99
Try("x".toInt).getOrElse(0)           // 0

// fold — handle both cases
result.fold(
  ex    => s"Failed: ${ex.getMessage}",
  value => s"Parsed: $value"
)

// Pattern matching
Try("abc".toInt) match {
  case Success(n)  => s"Got number: $n"
  case Failure(ex) => s"Parse error: ${ex.getMessage}"
}


## 3.1 Transforming `Try` with `map` and `flatMap`

Like `Option`, `Try` supports `map` and `flatMap`. A `Failure` propagates through any chain of transformations unchanged — you only write success-path logic.


In [ ]:
// map — transform Success, propagate Failure
Try("42".toInt).map(_ * 2)       // Success(84)
Try("x".toInt).map(_ * 2)        // Failure — unchanged

// Chain multiple transformations
val pipeline = Try("  15  ".trim.toInt)
  .map(_ * 2)         // 30
  .map(_ + 5)         // 35
  .map(n => s"Result: $n")

pipeline   // Success("Result: 35")

// If any step fails, the whole chain fails
val broken = Try("bad".toInt)
  .map(_ * 2)
  .map(_ + 5)
  .map(n => s"Result: $n")

broken   // Failure — propagated all the way through

In [ ]:
// flatMap — chain Try-returning functions
def parseInt(s: String): Try[Int]    = Try(s.trim.toInt)
def safeDivide(a: Int, b: Int): Try[Double] =
  if (b == 0) Failure(new ArithmeticException("Division by zero"))
  else Success(a.toDouble / b)
def safeSqrt(n: Double): Try[Double] =
  if (n < 0) Failure(new IllegalArgumentException(s"Cannot sqrt negative: $n"))
  else Success(Math.sqrt(n))

// Chain all three
val r1 = parseInt("100").flatMap(n => safeDivide(n, 4)).flatMap(safeSqrt)
val r2 = parseInt("100").flatMap(n => safeDivide(n, 0)).flatMap(safeSqrt)
val r3 = parseInt("abc").flatMap(n => safeDivide(n, 4)).flatMap(safeSqrt)

r1   // Success(5.0)
r2   // Failure(ArithmeticException: Division by zero)
r3   // Failure(NumberFormatException)


## 3.2 `for`-Comprehensions over `Try`


In [ ]:
// for-comprehension reads like sequential steps — any failure short-circuits
def compute(numStr: String, denStr: String): Try[String] =
  for {
    num    <- parseInt(numStr)
    den    <- parseInt(denStr)
    ratio  <- safeDivide(num, den)
    root   <- safeSqrt(ratio)
  } yield f"sqrt($num / $den) = $root%.4f"

compute("100", "4")    // Success("sqrt(100 / 4) = 5.0000")
compute("100", "0")    // Failure(division by zero)
compute("abc", "4")    // Failure(number format)
compute("100", "xyz")  // Failure(number format on second arg)


## 3.3 Recovery — `recover` and `recoverWith`

`recover` allows you to handle specific exceptions and replace a `Failure` with a `Success`. It is the `Try` equivalent of `getOrElse` but with access to the exception.


In [ ]:
// recover — handle a Failure and return a default Success
val recovered = Try("abc".toInt).recover {
  case _: NumberFormatException => 0
}
recovered   // Success(0)

// recoverWith — recover with another Try-returning expression
def parseIntOrDefault(s: String, default: => Int): Try[Int] =
  Try(s.toInt).recoverWith {
    case _: NumberFormatException => Success(default)
  }

parseIntOrDefault("42", 0)    // Success(42)
parseIntOrDefault("bad", -1)  // Success(-1)

// recover with pattern matching on exception type
val safe = Try(10 / 0).recover {
  case _: ArithmeticException  => Int.MaxValue
  case _: NullPointerException => -1
}
safe


## 3.4 `Try` for Parsing and I/O Wrappers

`Try` is the natural tool for wrapping any operation that may throw — parsing, file reading, JSON decoding, network calls, reflection.


In [ ]:
import java.time.LocalDate
import java.time.format.DateTimeParseException

// Safe date parsing
def parseDate(s: String): Try[LocalDate] = Try(LocalDate.parse(s.trim))

parseDate("2024-03-15")    // Success
parseDate("15/03/2024")    // Failure(DateTimeParseException)
parseDate("not-a-date")    // Failure

// Safe URL connection simulation
def fetchConfig(url: String): Try[Map[String, String]] = Try {
  if (!url.startsWith("http")) throw new IllegalArgumentException(s"Invalid URL: $url")
  // Simulate config fetching
  Map("host" -> "localhost", "port" -> "8080")
}

fetchConfig("http://config-server/app").map(cfg => s"host=${cfg("host")}")
fetchConfig("/etc/config").recover { case ex => s"Error: ${ex.getMessage}" }


# 4. `Either[E, A]` — Typed Error Channels

`Either[E, A]` is the most informative of the three error types. It is a sealed ADT with two cases:

- `Right(value: A)` — success (by convention, the "right" answer)
- `Left(error: E)` — failure, with a **typed error value of your own choosing**

```scala
sealed trait Either[+E, +A]
case class Right[A](value: A) extends Either[Nothing, A]
case class Left[E](error: E)  extends Either[E, Nothing]
```

The key difference from `Option` and `Try`:

| Type | Error information |
|------|------------------|
| `Option` | None — just absence |
| `Try` | A `Throwable` — JVM exception |
| `Either` | **Any type you define** — a string, an ADT, a case class |

`Either` is the idiomatic choice for **domain-level errors** — business rule violations, validation failures, and any situation where you want the error to carry structured data.


In [ ]:
// Creating Either values
val success: Either[String, Int] = Right(42)
val failure: Either[String, Int] = Left("Value not found")

success
failure

// Either with a typed error ADT
sealed trait AppError
case class NotFound(id: String)         extends AppError
case class InvalidInput(field: String, msg: String) extends AppError
case class DatabaseError(cause: String) extends AppError

def findProduct(id: String): Either[AppError, String] =
  if (id.startsWith("P")) Right(s"Product $id")
  else Left(NotFound(id))

findProduct("P001")   // Right("Product P001")
findProduct("X999")   // Left(NotFound("X999"))


## 4.1 Transforming `Either` with `map`, `flatMap`, `left.map`

`Either` is **right-biased** since Scala 2.12: `map` and `flatMap` operate on the `Right` value. `Left` values propagate unchanged.

To transform the `Left` value, use `.left.map`.


In [ ]:
val right: Either[String, Int] = Right(10)
val left:  Either[String, Int] = Left("error")

// map operates on Right
right.map(_ * 3)        // Right(30)
left.map(_ * 3)         // Left("error") — unchanged

// Chain transformations
right
  .map(_ + 5)           // Right(15)
  .map(_ * 2)           // Right(30)
  .map(n => s"val=$n")  // Right("val=30")

// left.map — transform the Left side
left.left.map(_.toUpperCase)   // Left("ERROR")
right.left.map(_.toUpperCase)  // Right(10) — unchanged

In [ ]:
// flatMap — chain Either-returning functions
def parseIntE(s: String): Either[String, Int] =
  s.trim.toIntOption.toRight(s"'$s' is not a valid integer")
  // toRight converts Option to Either: Some(x) => Right(x), None => Left(default)

def divideE(a: Int, b: Int): Either[String, Double] =
  if (b == 0) Left("Division by zero") else Right(a.toDouble / b)

def sqrtE(n: Double): Either[String, Double] =
  if (n < 0) Left(s"Cannot sqrt negative number: $n") else Right(Math.sqrt(n))

// Chain with flatMap
parseIntE("100").flatMap(n => divideE(n, 4)).flatMap(sqrtE)
parseIntE("100").flatMap(n => divideE(n, 0)).flatMap(sqrtE)
parseIntE("bad").flatMap(n => divideE(n, 4)).flatMap(sqrtE)


## 4.2 Extracting from `Either`


In [ ]:
val r: Either[String, Int] = Right(42)
val l: Either[String, Int] = Left("oops")

// fold — handle both sides
r.fold(err => s"Error: $err", v => s"Value: $v")  // "Value: 42"
l.fold(err => s"Error: $err", v => s"Value: $v")  // "Error: oops"

// getOrElse
r.getOrElse(0)   // 42
l.getOrElse(0)   // 0

// Pattern matching
r match {
  case Right(v)   => s"Success: $v"
  case Left(err)  => s"Failure: $err"
}

// isRight / isLeft
r.isRight   // true
l.isLeft    // true


## 4.3 Typed Error ADTs with `Either`

The real power of `Either` emerges when you combine it with a **sealed error ADT**. This lets the compiler enforce that every error case is handled, making error handling as exhaustive as any other pattern match.


In [ ]:
// Domain-specific error hierarchy
sealed trait ValidationError
case class EmptyField(fieldName: String)                    extends ValidationError
case class TooShort(fieldName: String, minLen: Int)         extends ValidationError
case class TooLong(fieldName: String, maxLen: Int)          extends ValidationError
case class InvalidFormat(fieldName: String, expected: String) extends ValidationError
case class OutOfRange(fieldName: String, min: Int, max: Int, actual: Int) extends ValidationError

type Validated[A] = Either[ValidationError, A]  // type alias for readability

// Validation functions — each returns Validated
def validateName(name: String): Validated[String] =
  if (name.trim.isEmpty)        Left(EmptyField("name"))
  else if (name.trim.length < 2) Left(TooShort("name", 2))
  else if (name.trim.length > 50) Left(TooLong("name", 50))
  else Right(name.trim.capitalize)

def validateAge(s: String): Validated[Int] =
  s.trim.toIntOption match {
    case None    => Left(InvalidFormat("age", "integer"))
    case Some(n) if n < 0 || n > 150 => Left(OutOfRange("age", 0, 150, n))
    case Some(n) => Right(n)
  }

def validateEmail(email: String): Validated[String] = {
  val e = email.trim.toLowerCase
  if (e.isEmpty) Left(EmptyField("email"))
  else if (!e.contains("@") || !e.contains(".")) Left(InvalidFormat("email", "user@domain.tld"))
  else Right(e)
}

// Pattern match on error ADT — compiler checks exhaustiveness
def describeError(e: ValidationError): String = e match {
  case EmptyField(f)           => s"Field '$f' must not be empty"
  case TooShort(f, min)        => s"Field '$f' must be at least $min characters"
  case TooLong(f, max)         => s"Field '$f' must not exceed $max characters"
  case InvalidFormat(f, exp)   => s"Field '$f' must be a valid $exp"
  case OutOfRange(f, mn, mx, v) => s"Field '$f': $v is out of range [$mn, $mx]"
}

List(
  validateName(""),
  validateName("A"),
  validateName("Alice"),
  validateAge("abc"),
  validateAge("200"),
  validateAge("30"),
  validateEmail("bad"),
  validateEmail("ok@test.com")
).map(_.left.map(describeError))


## 4.4 `for`-Comprehensions over `Either`


In [ ]:
case class RegisterRequest(name: String, ageStr: String, email: String)
case class NewUser(name: String, age: Int, email: String)

// Chain validations with for-comprehension — first failure short-circuits
def registerUser(req: RegisterRequest): Either[ValidationError, NewUser] =
  for {
    name  <- validateName(req.name)
    age   <- validateAge(req.ageStr)
    email <- validateEmail(req.email)
  } yield NewUser(name, age, email)

// Test cases
registerUser(RegisterRequest("Alice", "30", "alice@example.com")).map(u => s"Registered: $u")
registerUser(RegisterRequest("", "30", "alice@example.com")).left.map(describeError)
registerUser(RegisterRequest("Bob", "abc", "bob@example.com")).left.map(describeError)
registerUser(RegisterRequest("Charlie", "25", "not-valid")).left.map(describeError)


# 5. Converting Between `Option`, `Try`, and `Either`

In real codebases you often need to move between these three types. Scala provides direct conversion methods.


In [ ]:
import scala.util.{Try, Success, Failure}

// Option => Either
val opt: Option[Int] = Some(42)
val noneOpt: Option[Int] = None

opt.toRight("Value was missing")      // Right(42)
noneOpt.toRight("Value was missing")  // Left("Value was missing")
opt.toLeft("fallback")                // Left(42)  — swapped: Some becomes Left

In [ ]:
// Try => Option (exception info is lost)
Try("42".toInt).toOption    // Some(42)
Try("x".toInt).toOption     // None

// Try => Either (exception becomes Left)
Try("42".toInt).toEither    // Right(42)
Try("x".toInt).toEither     // Left(NumberFormatException)

// Either => Option (Left is discarded)
Right(42).toOption          // Some(42)
Left("error").toOption      // None

// Option => Try
Some(42).toRight(new NoSuchElementException("missing")) // not direct — use toEither then map
// Practical pattern:
def optToTry[A](opt: Option[A], msg: String): Try[A] =
  opt match {
    case Some(v) => Success(v)
    case None    => Failure(new NoSuchElementException(msg))
  }

optToTry(Some(10), "not found")
optToTry(None, "not found")

In [ ]:
// Conversion table summary
val num = Try("5".toInt)

val asOption: Option[Int]            = num.toOption
val asEither: Either[Throwable, Int] = num.toEither
val backToTry: Try[Int]              = asEither.toTry  // Either[Throwable, A] => Try[A]
val eitherToOpt: Option[Int]         = asEither.toOption

asOption
asEither
backToTry
eitherToOpt


## When to use which type

| Scenario | Use |
|----------|---------|
| Value may simply be absent — absence is not an error | `Option[A]` |
| Wrapping Java code, parsing, I/O that may throw | `Try[A]` |
| Domain errors with typed, structured information | `Either[E, A]` |
| Multiple independent validations to accumulate | `Either` + `ValidatedNel` (Cats) |

A useful mental rule: **`Option` for absence, `Try` for exceptions, `Either` for domain errors.**



# 6. Accumulating Multiple Errors

A critical limitation of `for`-comprehensions over `Either` is that they **short-circuit** — the first `Left` stops the chain. This means you only ever report one error at a time.

For form validation and batch processing, you usually want to **collect all errors at once**. The standard Scala approach is to run validations independently and then combine the results.


In [ ]:
// Run all validations and collect ALL errors
case class UserForm(name: String, ageStr: String, email: String)
case class ValidatedUser(name: String, age: Int, email: String)

def validateAllFields(form: UserForm): Either[List[ValidationError], ValidatedUser] = {
  // Run each validation independently
  val nameResult  = validateName(form.name)
  val ageResult   = validateAge(form.ageStr)
  val emailResult = validateEmail(form.email)

  // Collect all errors
  val errors = List(nameResult, ageResult, emailResult).collect {
    case Left(err) => err
  }

  if (errors.nonEmpty) Left(errors)
  else Right(ValidatedUser(
    nameResult.getOrElse(""),
    ageResult.getOrElse(0),
    emailResult.getOrElse("")
  ))
}

// A form with MULTIPLE errors — all are reported at once
val badForm = UserForm("", "abc", "not-an-email")
validateAllFields(badForm).left.map(_.map(describeError))

// A valid form
val goodForm = UserForm("alice", "28", "alice@example.com")
validateAllFields(goodForm)


For even more powerful error accumulation (with full applicative composition), Cats' `Validated` type is the standard approach in production Scala. The pattern above is the standard-library approximation.



# 7. Pure Functions and Error Handling — Core Principles

A **pure function** has two properties:

1. **Totality**: it returns a value for every possible input
2. **Referential transparency**: it has no side effects — calling it twice with the same input always yields the same output

Throwing an exception violates both properties: the function is not total (it doesn't return for bad input), and the thrown exception is a side effect observable outside the function.

By returning `Option`, `Try`, or `Either`, a function becomes **total** — it always returns something — and the failure is a **value**, not a side effect.


In [ ]:
// IMPURE: throws, has side effects, partial
def lookupImpure(id: Int): String = {
  val db = Map(1 -> "Alice")
  if (!db.contains(id)) throw new RuntimeException(s"User $id not found")
  db(id)
}

// PURE: total, referentially transparent, error is a value
def lookupPure(id: Int): Either[String, String] = {
  val db = Map(1 -> "Alice")
  db.get(id).toRight(s"User $id not found")
}

lookupPure(1)    // Right("Alice")
lookupPure(99)   // Left("User 99 not found")

// The caller is FORCED to handle both cases — no surprises
lookupPure(1).fold(
  err  => println(s"Handled error: $err"),
  user => println(s"Found user: $user")
)


# 8. Complete Data Processing Pipeline

Now we assemble everything into a realistic **CSV data ingestion pipeline** using only pure functions with `Option`, `Try`, and `Either`.

The pipeline:
1. Parse raw CSV rows (`Try` — may throw on malformed input)
2. Validate each field (`Either` — typed domain errors)
3. Enrich with lookup data (`Option` — optional enrichment)
4. Aggregate results (pure `foldLeft`)
5. Report successes and failures without a single `throw` or `null`


In [ ]:
import scala.util.{Try, Success, Failure}

// =========================================
// Domain Model
// =========================================

sealed trait ParseError
case class MalformedRow(row: String, reason: String)     extends ParseError
case class InvalidField(field: String, value: String, reason: String) extends ParseError
case class UnknownReference(field: String, ref: String)  extends ParseError

case class RawRow(fields: Map[String, String])

case class Transaction(
  id:          String,
  accountId:   String,
  accountName: Option[String],  // enriched from lookup
  amount:      BigDecimal,
  currency:    String,
  date:        String
)

In [ ]:
// =========================================
// Stage 1: Parse CSV row using Try
// =========================================

def parseRow(rawLine: String): Try[RawRow] = Try {
  if (rawLine.trim.isEmpty) throw new IllegalArgumentException("Empty row")
  val parts = rawLine.split(",", -1).map(_.trim)
  val headers = List("id", "account_id", "amount", "currency", "date")
  if (parts.length != headers.length)
    throw new IllegalArgumentException(
      s"Expected ${headers.length} fields, got ${parts.length}"
    )
  RawRow(headers.zip(parts).toMap)
}

// Test parsing
parseRow("TXN001, ACC42, 199.99, USD, 2024-03-15")
parseRow("bad data here")   // Failure
parseRow("")                // Failure

In [ ]:
// =========================================
// Stage 2: Validate fields using Either
// =========================================

// Lookup table for account names (simulates a database)
val accountNames: Map[String, String] = Map(
  "ACC42"  -> "Acme Corporation",
  "ACC17"  -> "Beta Industries",
  "ACC99"  -> "Gamma LLC"
)

def requireField(row: RawRow, field: String): Either[ParseError, String] =
  row.fields.get(field)
    .filter(_.nonEmpty)
    .toRight(InvalidField(field, "", "field is missing or empty"))

def parseAmount(s: String): Either[ParseError, BigDecimal] =
  Try(BigDecimal(s)).toEither
    .left.map(_ => InvalidField("amount", s, "must be a valid decimal number"))
    .flatMap { bd =>
      if (bd <= 0) Left(InvalidField("amount", s, "must be positive"))
      else Right(bd)
    }

val validCurrencies = Set("USD", "EUR", "GBP", "TRY")

def validateCurrency(s: String): Either[ParseError, String] =
  if (validCurrencies.contains(s.toUpperCase)) Right(s.toUpperCase)
  else Left(InvalidField("currency", s, s"must be one of ${validCurrencies.mkString(", ")}"))

def validateDate(s: String): Either[ParseError, String] = {
  import java.time.LocalDate
  Try(LocalDate.parse(s)).toEither
    .left.map(_ => InvalidField("date", s, "must be in YYYY-MM-DD format"))
    .map(_.toString)
}

// Validate the entire row using for-comprehension
def validateRow(row: RawRow): Either[ParseError, Transaction] =
  for {
    id       <- requireField(row, "id")
    accId    <- requireField(row, "account_id")
    amtStr   <- requireField(row, "amount")
    amount   <- parseAmount(amtStr)
    currStr  <- requireField(row, "currency")
    currency <- validateCurrency(currStr)
    dateStr  <- requireField(row, "date")
    date     <- validateDate(dateStr)
  } yield Transaction(
    id          = id,
    accountId   = accId,
    accountName = accountNames.get(accId),  // Option: None if not found, no error
    amount      = amount,
    currency    = currency,
    date        = date
  )

In [ ]:
// =========================================
// Stage 3: The Full Pipeline
// Compose Try (parsing) with Either (validation)
// =========================================

def processLine(line: String): Either[ParseError, Transaction] =
  parseRow(line)
    .toEither
    .left.map(ex => MalformedRow(line, ex.getMessage))
    .flatMap(validateRow)

// Sample input data
val csvLines = List(
  "TXN001, ACC42, 199.99, USD, 2024-03-15",
  "TXN002, ACC17, 45.00,  EUR, 2024-03-16",
  "TXN003, ACC99, -50.00, GBP, 2024-03-16",  // negative amount
  "TXN004, ACC77, 100.00, USD, 2024-03-17",  // unknown account (ok — enriched as None)
  "TXN005, ACC42, 75.00,  XYZ, 2024-03-17",  // invalid currency
  "TXN006, ACC17, abc,    USD, 2024-03-18",  // invalid amount
  "TXN007, ACC99, 250.00, EUR, not-a-date",  // invalid date
  "bad row without enough commas",             // malformed structure
  "TXN009, ACC42, 300.00, GBP, 2024-03-20",
  ""
)

val results: List[Either[ParseError, Transaction]] = csvLines.map(processLine)

In [ ]:
// =========================================
// Stage 4: Separate successes from failures
// =========================================

val successes: List[Transaction] = results.collect { case Right(t) => t }
val failures:  List[ParseError]  = results.collect { case Left(e)  => e }

println(s"Processed ${results.length} rows")
println(s"Successes: ${successes.length}")
println(s"Failures:  ${failures.length}")

println("\n=== Successful Transactions ===")
successes.foreach { t =>
  val name = t.accountName.getOrElse("[Unknown Account]")
  println(f"${t.id}  $name%-22s  ${t.amount}%8.2f ${t.currency}  ${t.date}")
}

println("\n=== Failed Rows ===")
failures.foreach {
  case MalformedRow(row, reason) =>
    println(s"  MALFORMED: '$row' — $reason")
  case InvalidField(field, value, reason) =>
    println(s"  INVALID FIELD '$field': '$value' — $reason")
  case UnknownReference(field, ref) =>
    println(s"  UNKNOWN REF '$field': '$ref'")
}

In [ ]:
// =========================================
// Stage 5: Aggregate successful transactions
// Pure foldLeft — no mutation, no exceptions
// =========================================

case class Summary(
  totalAmount: BigDecimal,
  countByCurrency: Map[String, Int],
  countByAccount: Map[String, Int],
  enrichedCount: Int
)

val summary = successes.foldLeft(
  Summary(BigDecimal(0), Map.empty, Map.empty, 0)
) { (acc, t) =>
  Summary(
    totalAmount      = acc.totalAmount + t.amount,
    countByCurrency  = acc.countByCurrency.updated(
                         t.currency,
                         acc.countByCurrency.getOrElse(t.currency, 0) + 1
                       ),
    countByAccount   = acc.countByAccount.updated(
                         t.accountId,
                         acc.countByAccount.getOrElse(t.accountId, 0) + 1
                       ),
    enrichedCount    = acc.enrichedCount + t.accountName.fold(0)(_ => 1)
  )
}

println("\n=== Pipeline Summary ===")
println(f"Total amount: ${summary.totalAmount}%.2f (mixed currencies)")
println(s"Transactions by currency: ${summary.countByCurrency}")
println(s"Transactions by account:  ${summary.countByAccount}")
println(s"Enriched with account name: ${summary.enrichedCount} of ${successes.length}")


This pipeline demonstrates every principle from this lecture working together:

| Technique | Where used |
|-----------|------------|
| `Try` | `parseRow` — wraps exception-throwing string splitting |
| `Try.toEither` | Converts parse exception into typed `ParseError` |
| `Either` with ADT errors | All field validation functions |
| `for`-comprehension over `Either` | `validateRow` — chains validations |
| `Option` for enrichment | `accountNames.get(accId)` — absence is not an error |
| `Option.fold` | Rendering account name with fallback |
| `Option.getOrElse` | Rendering `[Unknown Account]` |
| `flatMap` composition | `processLine` chains `Try` parse then `Either` validation |
| `collect` | Splitting results into successes and failures |
| `foldLeft` | Pure aggregation with no mutation |
| Pattern matching on error ADT | Structured failure reporting |
| Pure functions throughout | No `throw`, no `null`, no side effects in business logic |



# Final Summary

### The Three Types at a Glance

| Type | Cases | Represents | Use When |
|------|-------|------------|----------|
| `Option[A]` | `Some(a)` / `None` | Presence or absence | Value may not exist; absence is not an error |
| `Try[A]` | `Success(a)` / `Failure(ex)` | Success or exception | Wrapping exception-throwing code (parsing, I/O) |
| `Either[E, A]` | `Right(a)` / `Left(e)` | Typed success or failure | Domain errors with structured, typed information |

### Core Operations

| Operation | `Option` | `Try` | `Either` |
|-----------|----------|-------|----------|
| Transform success | `map` | `map` | `map` (right-biased) |
| Chain fallible ops | `flatMap` | `flatMap` | `flatMap` |
| Default on failure | `getOrElse` | `getOrElse` | `getOrElse` |
| Handle both cases | `fold` | `fold` | `fold` |
| Recover from failure | `orElse` | `recover` / `recoverWith` | `left.map` |
| Filter value | `filter` | — | `filterOrElse` |
| Sequential chain | `for`-comprehension | `for`-comprehension | `for`-comprehension |

### Conversion Cheat Sheet

| From | To | Method |
|------|----|--------|
| `Option[A]` | `Either[E, A]` | `.toRight(leftValue)` |
| `Try[A]` | `Option[A]` | `.toOption` |
| `Try[A]` | `Either[Throwable, A]` | `.toEither` |
| `Either[Throwable, A]` | `Try[A]` | `.toTry` |
| `Either[E, A]` | `Option[A]` | `.toOption` |

### Principles of Pure Functional Error Handling

1. **Totality** — functions return a value for every input; failure is a value, not an exception
2. **Explicit errors** — the return type tells the caller what can go wrong
3. **Composability** — `map`, `flatMap`, and `for`-comprehensions chain error-prone operations cleanly
4. **No `null`** — `Option` replaces every nullable reference
5. **No `throw` in business logic** — `Try` captures exceptions at boundaries; `Either` expresses domain errors
6. **Compiler enforcement** — sealed error ADTs with pattern matching ensure all failure cases are handled

Mastering these three types is essential for writing safe, idiomatic Scala in **data pipelines, microservices, domain modeling, Apache Spark transformations, and every modern Scala application**.
